In [ ]:
import time
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [ ]:
# Task 1: Data Acquisition via Web Scraping
# Extract raw T20I cricket player performance data (batting and bowling) from ESPN Cricinfo.
# We will scrape multiple years of data, parse the HTML tables, and compile them into a structured dataset.

In [ ]:
import time, requests
import pandas as pd
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter, Retry

# Target URL and extraction parameters
BASE = "https://stats.espncricinfo.com/ci/engine/stats/index.html"
YEARS, MAX_PAGES, DELAY = range(2015, 2026), 4, 2.5

In [ ]:
# 1.1 Session Configuration & Error Handling
# Web scraping scripts are often blocked by website servers. To ensure our script runs uninterrupted,
# we configure a robust HTTP session using standard browser headers and an automatic retry strategy for network drops.

In [ ]:
# Initialize session with browser headers to mimic human traffic
session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"})

# Apply automatic retries for common server errors (e.g., Too Many Requests)
retry = Retry(total=5, backoff_factor=2, status_forcelist=[403, 429, 500, 502, 503, 504])
session.mount("https://", HTTPAdapter(max_retries=retry))

In [ ]:
# 1.2 HTML Parsing & Formatting Logic
# The raw HTML contains unstructured text (e.g., player names and team abbreviations in the same cell). 
# We define helper functions to cleanly separate these strings and safely request the page HTML.

In [ ]:
def player_team(text):
    """Separates player name and team from format: 'Virat Kohli (IND)'"""
    text = text.replace("\xa0", " ").strip()
    if "(" in text and text.endswith(")"):
        name, team = text.rsplit("(", 1)
        return name.strip(), team[:-1].strip()
    return text, ""

def get_soup(stat_type, year, page):
    """Fetches and parses the HTML for a specific page and year."""
    url = f"{BASE}?class=3;template=results;type={stat_type};spanmin1=01+Jan+{year};spanmax1=31+Dec+{year};spanval1=span;page={page}"
    response = session.get(url, timeout=30)
    return BeautifulSoup(response.text, "html.parser") if response.status_code == 200 else None

In [ ]:
# 1.3 Execution & Data Export
# A loop through our defined years and pages to extract both batting and bowling tables. Finally, 
# we merge the two datasets based on the player's name and year, exporting the final raw data to a CSV for Task 2 (Data Cleaning).

In [ ]:
def scrape_data(stat_type):
    """Iterates through years and pages to extract player data."""
    records = []
    for year in YEARS:
        for page in range(1, MAX_PAGES + 1):
            soup = get_soup(stat_type, year, page)
            time.sleep(DELAY) # Pause to prevent overloading the server
            if not soup: continue
            
            # Find the main data table and extract rows
            table = soup.find("table", class_="engineTable")
            if not table: continue
            
            for tr in table.find_all("tr", class_="data1"):
                cells = [td.get_text(" ", strip=True) for td in tr.find_all("td")]
                if len(cells) >= 4:
                    player, team = player_team(cells[0])
                    records.append({"player": player, "team": team, "year": year, "matches": cells[1], "runs_or_wkts": cells[4]})
    return pd.DataFrame(records)

# NOTE: The execution is commented out to prevent re-scraping the server accidentally. 
# bat_df = scrape_data("batting")
# bowl_df = scrape_data("bowling")
# df = pd.merge(bat_df, bowl_df, on=["player", "team", "year"], how="outer")
# df.to_csv("Web_t20i_players.csv", index=False)
# print("Scraping complete and saved!")

In [ ]:
# Load the dataset into the notebook
df = pd.read_csv(r"C:\Users\dell\Web_t20i_players.csv")
print("Data loaded successfully! Shape:", df.shape)

In [ ]:
# checking the columns and data types
df.info()

In [ ]:
# taking a quick look at the actual data
display(df.head(1000))